### Build the African Language Confusion Prompt Set

This notebook builds the prompt dataset for the studys

| Source | Task | Languages | Target count/language |
|---|---|---|---|
| Aya | monolingual | 10 | 1000 |
| Dolly | monolingual | 8 | 200 |
| PolyWrite | monolingual | 17 | 100 |
| AfriQA | monolingual | 8 | 1000 |
| Okapi | crosslingual | 18 (union of all monolingual languages) | 100 |
| ShareGPT | crosslingual | 18 | 1000 |

**Monolingual** = a prompt written natively in the target language, expecting a reply in
that language. **Crosslingual** = an English instruction asking the model to reply in a
target African language (built by re-templating the English base prompts from the
language-confusion benchmark's Okapi/ShareGPT crosslingual sets, substituting the target
language name into the same `"Reply in {language}."` / `"... Write in {language}."`
instruction patterns).

**Note on Aya coverage:** Fon, Twi and Kinyarwanda are under Aya, but the
`CohereForAI/aya_dataset` split used here does not
contain any rows for those three languages -- confirmed by inspecting the full
`language` value_counts (73 languages, none of them Fon/Twi/Kinyarwanda). They exist only
in the larger, machine-templated `aya_collection_language_split`, which is a different
kind of data (templated NLP tasks vs. organic human prompts) and was deliberately not
mixed in. Fon, Twi and Kinyarwanda are still covered by PolyWrite/AfriQA and by the
crosslingual sets -- see `languages.py` for the full registry and rationale.

Output: one CSV per (task, source, language) under `prompts/`, plus a single consolidated `prompts/all_prompts.csv`
with columns `id, prompt, source, task, language`. The `Open_Source_Models.ipynb` and
`Closed_Source_Models.ipynb` notebooks read `all_prompts.csv` to run the model sweeps.

### Setup

Run once, from the `african-language-confusion/` folder, in a virtual environment (keeps
these dependencies out of your global Python install):

```bash
python -m venv venv
venv\Scriptsctivate      # Windows
source venv/bin/activate    # macOS/Linux
pip install -r requirements.txt
```

Then pick `venv` as this notebook's kernel before running the cells below.

Optional: copy `.env.example` to `.env` and set `HF_TOKEN` (from
https://huggingface.co/settings/tokens) to authenticate Hugging Face Hub requests --
avoids the "unauthenticated requests" rate-limit warning below and speeds up downloads.
Without it, the Aya/Dolly/PolyWrite loads below still work anonymously.

In [1]:
import prompts
import languages

print(f"{len(languages.LANGUAGES)} languages in the registry:")
for key, info in languages.LANGUAGES.items():
    sources = [s for s in ("aya", "dolly", "polywrite", "afriqa") if s in info]
    print(f"  {key:14s} ({info['name']:16s}) monolingual sources: {', '.join(sources) or '(none -- crosslingual target only)'}")


18 languages in the registry:
  acholi         (Acholi          ) monolingual sources: polywrite
  amharic        (Amharic         ) monolingual sources: aya, dolly, polywrite
  chichewa       (Chichewa        ) monolingual sources: aya, polywrite
  fongbe         (Fongbe          ) monolingual sources: polywrite, afriqa
  ga             (Ga              ) monolingual sources: polywrite
  hausa          (Hausa           ) monolingual sources: aya, dolly, polywrite, afriqa
  igbo           (Igbo            ) monolingual sources: aya, dolly, polywrite, afriqa
  kinyarwanda    (Kinyarwanda     ) monolingual sources: polywrite, afriqa
  lingala        (Lingala         ) monolingual sources: polywrite
  malagasy       (Malagasy        ) monolingual sources: aya, dolly, polywrite
  ndebele        (Ndebele         ) monolingual sources: polywrite
  sepedi         (Northern Sotho  ) monolingual sources: aya, dolly, polywrite
  shona          (Shona           ) monolingual sources: aya, dolly, 

### Build all sources

This downloads each dataset (public, no API keys needed), filters to our language
registry, samples up to the target count per language (warning -- not silently
truncating -- when a language has fewer prompts available than requested), and writes
the CSVs. Takes a few minutes, mostly spent downloading PolyWrite (~35k rows) and the
LCB `test_sets.zip` (for the crosslingual base prompts).


In [2]:
df = prompts.build_all_prompts()
prompts.save_test_sets(df, out_dir="prompts")
df.shape


Loading aya...
Loading dolly...
Loading polywrite...


Resolving data files:   0%|          | 0/240 [00:00<?, ?it/s]

  [WARN] polywrite/ga: requested 100, only 74 available -- using all of them
Loading afriqa...
Loading crosslingual_okapi...
Loading crosslingual_sharegpt...
  crosslingual_sharegpt_base: 55365 English turns kept, 3376 non-English turns dropped
Saved 11774 prompts to prompts/ (2 tasks, 6 sources, 18 languages)


(11774, 5)

### Prompt Statistics

In [3]:
summary = (
    df.groupby(["task", "source", "language"])
    .size()
    .rename("n_prompts")
    .reset_index()
    .pivot_table(index=["task", "source"], values="n_prompts", aggfunc=["sum", "mean", "min", "max"])
)
summary


sum         mean       min       max
                       n_prompts    n_prompts n_prompts n_prompts
task         source                                              
crosslingual okapi          1800   100.000000       100       100
             sharegpt       3600   200.000000       200       200
monolingual  afriqa         8000  1000.000000      1000      1000
             aya            1000   100.000000       100       100
             dolly          1600   200.000000       200       200
             polywrite      1774    98.555556        74       100

### Spot-check a few rendered prompts per source

In [3]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

for source in df["source"].unique():
    sample = df[df["source"] == source].sample(2, random_state=0)
    print(f"=== {source} ===")
    for _, row in sample.iterrows():
        print(f"[{row['language']}] {row['prompt']}")
    print()


=== aya ===
[yoruba] Àṣà ilẹ̀ Áfríkà
[wolof] Fann mooy gëblagréewum Inde?

=== dolly ===
[shona] Sarudza kana izvi zvingava zvinobetsera kana kuti zvisingabetseri kuti mudzidzi wechikoro chapamusoro azviise muhomwe yake. Mabhuku, bhuku rokudzidza, rambi repatafura, homwe yemapenzura, bhora rokumahombekombe, mutsago, laptop.
[hausa] Menene laser kuma wanene ya ƙirƙira shi?
Context:Laser na'ura ce da ke fitar da haske ta hanyar aiwatar da fadada gani wanda ya danganci fitowar fitowar hasken lantarki. Kalmar laser wani abu ne wanda ya samo asali ne a matsayin acronym don fadada haske ta hanyar motsawar radiation. An gina laser na farko a 1960 da Theodore Maiman a Hughes Research Laboratories, bisa ga aikin da Charles H. Townes da Arthur Leonard Schawlow suka yi.  Laser ya bambanta da sauran tushen haske a yadda yake fitar da haske da ke da daidaito. Haɗin sararin samaniya yana ba da damar mayar da hankali ga laser zuwa wani wuri mai mahimmanci, yana ba da damar aikace-aikace kamar yankan 